# TSDF fusion — depth maps → a 3D mesh

**Track D · Scene & World Models** · maps to lesson D3 (TSDF fusion).

Dense reconstruction integrates many depth images into a **Truncated Signed Distance** volume; the surface is its zero level set. We simulate depth from six viewpoints of a shape, fuse, and run marching cubes.

> CPU is fine.

In [ ]:
import os, numpy as np, matplotlib.pyplot as plt
R = int(os.environ.get("STEPS", 64)); tau = 3.0 / R * 4   # grid res, truncation
g = np.linspace(-1, 1, R)
X, Y, Z = np.meshgrid(g, g, g, indexing="ij")

## 1 · The scene (two spheres) and its occupancy

In [ ]:
def sdf(x, y, z):
    d1 = np.sqrt((x + .25) ** 2 + y ** 2 + z ** 2) - 0.55
    d2 = np.sqrt((x - .45) ** 2 + (y - .2) ** 2 + z ** 2) - 0.4
    return np.minimum(d1, d2)
occ = sdf(X, Y, Z) < 0
print("occupied voxels:", int(occ.sum()))

## 2 · Simulate 6 orthographic depth views and fuse into a TSDF

In [ ]:
tsdf = np.zeros((R, R, R)); wsum = np.zeros((R, R, R))
axes = [(0, 1), (0, -1), (1, 1), (1, -1), (2, 1), (2, -1)]   # (axis, direction)
coord = [X, Y, Z]
for ax, sign in axes:
    o = occ if sign > 0 else occ[tuple(slice(None, None, -1) if i == ax else slice(None) for i in range(3))]
    c = coord[ax] if sign > 0 else coord[ax][tuple(slice(None, None, -1) if i == ax else slice(None) for i in range(3))]
    # nearest surface coordinate seen by this view, per (u,v) column along 'ax'
    first = np.argmax(o, axis=ax); seen = o.any(axis=ax)
    surf = np.take_along_axis(c, np.expand_dims(first, ax), axis=ax)            # (.. surface coord ..)
    raw = c - surf                                                              # +ve in front of surface, -ve behind
    w = (np.abs(raw) < tau) & np.expand_dims(seen, ax)
    raw = np.clip(raw, -tau, tau)
    if sign < 0:  # undo the flip
        raw = raw[tuple(slice(None, None, -1) if i == ax else slice(None) for i in range(3))]
        w = w[tuple(slice(None, None, -1) if i == ax else slice(None) for i in range(3))]
    tsdf += np.where(w, raw, 0.0); wsum += w
vol = np.where(wsum > 0, tsdf / np.maximum(wsum, 1), tau)
print("TSDF range:", round(float(vol.min()), 3), "to", round(float(vol.max()), 3))

## 3 · Extract the surface (marching cubes)

In [ ]:
from skimage import measure
verts, faces, _, _ = measure.marching_cubes(vol, level=0.0)
fig = plt.figure(figsize=(5, 5)); ax = fig.add_subplot(111, projection="3d")
ax.plot_trisurf(verts[:, 0], verts[:, 1], verts[:, 2], triangles=faces, cmap="viridis", lw=0)
ax.set_title(f"fused mesh — {verts.shape[0]} verts"); ax.set_axis_off(); plt.show()

## Save artifacts — checkpoint · metrics · figure
Everything is written to **outputs/D_tsdf_fusion/** — the model checkpoint, the full loss/eval history as JSON, and the final figure — then zipped. Colab sessions are ephemeral, so the last lines show how to download the zip or copy it to Google Drive.

In [ ]:
import os, json, torch, shutil
run = "outputs/D_tsdf_fusion"; os.makedirs(run, exist_ok=True)
torch.save({"verts": torch.tensor(verts.copy()), "faces": torch.tensor(faces.copy())}, f"{run}/mesh.pt")
json.dump({"verts": int(verts.shape[0]), "faces": int(faces.shape[0])}, open(f"{run}/metrics.json", "w"), indent=2)
try:
    fig.savefig(f"{run}/figure.png", dpi=120, bbox_inches="tight")
except Exception: pass
shutil.make_archive(run, "zip", run)
print("saved to", run, "->", sorted(os.listdir(run)))
# keep it past the session:  from google.colab import files; files.download(f"{run}.zip")
# or mount Drive:  from google.colab import drive; drive.mount('/content/drive')  # then shutil.copytree(run, "/content/drive/MyDrive/"+run)

## (Optional) Publish to the Hugging Face Hub
Upload this run as a **model repo** — the checkpoint, `metrics.json` (full loss/eval history) and the results figure, embedded in an auto-generated model card. Do it for each lab, then group them into a **Collection** on your HF profile (Profile → New collection), or with the commented `add_collection_item` call below. It needs a **write token**, so it only runs once you sign in.

In [ ]:
# (optional) publish this run as a Hugging Face model repo — checkpoint + metrics + plot
!pip -q install huggingface_hub
from huggingface_hub import HfApi, notebook_login
import os
notebook_login()   # paste a WRITE token from https://huggingface.co/settings/tokens
api = HfApi(); user = api.whoami()["name"]
lab = os.path.basename(run); repo_id = f"{user}/" + lab.lower().replace("_", "-")
fig = "\n![results](figure.png)\n" if os.path.exists(f"{run}/figure.png") else ""
open(f"{run}/README.md", "w").write("---\ntags: [ropedia-academy, education]\n---\n# " + lab + "\n\nTrained in **Ropedia Academy** (educational lab). Checkpoint, full loss/eval history (metrics.json) and the results figure are included." + fig)
api.create_repo(repo_id, repo_type="model", exist_ok=True)
api.upload_folder(folder_path=run, repo_id=repo_id, commit_message="Upload from Ropedia Academy")
print("uploaded ->", "https://huggingface.co/" + repo_id)
# group everything into one Collection (run once, after a few uploads):
# from huggingface_hub import create_collection, add_collection_item
# col = create_collection("Ropedia Academy - trained models", namespace=user, exists_ok=True)
# add_collection_item(col.slug, item_id=repo_id, item_type="model")

## How this links to tracks A–D
Fuses **B · 3D** geometry into a scene map.

### Where to go next
- Use real RGB-D (a phone LiDAR scan, RealSense) with true camera poses; weight by depth confidence.
- Stream it online with pose tracking → dense **SLAM** (the SplaTAM advanced lab).